# GraphRAG

## Import packages

In [22]:
import sys
sys.path.append('..')
sys.path.append('../neurorag')
sys.path.append('../neurorag/chains')

import os
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import json
from dotenv import load_dotenv
from getpass import getpass

from neurorag.neurorag import NeuroRAG

from metrics import (
  embeddings_cosine_sim_metric,
  bleu_metric,
  rogue_l_metric,
  rogue_1_metric,
  factscore_metric,
  summac_zs_metric,
  summac_conv_metric,
)

## Disable warnings

In [23]:
import warnings
warnings.filterwarnings('ignore')

## Setup environment variables

You have to define the following environment variables in the `.env` file, terminal environment, or input field within this Jupyter notebook.

## Import packages

In [24]:
env_variables = [
  'TAVILY_API_KEY',
  'ENTREZ_EMAIL',
  'OPENROUTER_API_KEY',
  'CHROMA_API_KEY',
  'CHROMA_TENANT',
  'CHROMA_DATABASE',
  'CHROMA_COLLECTION_NAME',
]

load_dotenv()

for key in env_variables:
  value = os.getenv(key)

  if value is None:
    value = getpass(key)

  os.environ[key] = value

## Build model

In [25]:
app = NeuroRAG(debug=True)
app.compile()

## Evaluate RAG

### Load QA dataset

In [26]:
mediqa_df = pd.read_csv('../datasets/pubmed_summary_qa.csv')
mediqa_df

,question,answer
0,What is the significance of the inferior prefr...,"The inferior prefrontal cortex, including area..."
1,How do circadian activity rhythms affect cogni...,Consistent circadian activity rhythms are asso...
2,How do executive function deficits affect read...,Executive function deficits can affect reading...
3,How does brain activity differ between highly ...,Highly hypnotizable individuals tend to exhibi...
4,What is the role of predictive neural activity...,Predictive neural activity plays a central rol...
5,How is serotonin signalling related to aggress...,Serotonin signalling has been implicated in th...
6,What brain regions are involved in motor urgency?,"The cerebellum, sensorimotor cortex, and prefr..."
7,How does bilingualism affect cognitive develop...,Bilingualism has been shown to influence cogni...
8,What is the role of the insula in major depres...,The insula is thought to contribute to the pat...
9,How does bilingualism affect cognitive process...,Bilingualism can influence cognitive processin...


### Load cached RAGs responses

In [27]:
cache_path = Path('cache.json')

if not os.path.exists(cache_path):
  data = {}
  with open(cache_path, 'w') as file:
    json.dump(data, file)

with open(cache_path, 'r') as f:
  cache = json.load(f)

CACHE_KEY = 'text-to-text-neurorag-evaluation'

if CACHE_KEY not in cache:
  cache[CACHE_KEY] = {}

len(cache.keys())

3

In [28]:
questions = list(mediqa_df['question'].tolist())
expected_answers = list(mediqa_df['answer'].tolist())
predicted_answers = []

for index, question in tqdm(enumerate(questions)):
  if question not in cache[CACHE_KEY]:
    cache[CACHE_KEY][question] = app.invoke(question)['generation']

  predicted_answers.append(cache[CACHE_KEY][question])

  with open(cache_path, 'w') as f:
    json.dump(cache, f)

cos_score = embeddings_cosine_sim_metric(expected_answers, predicted_answers)
bleu_score = bleu_metric(expected_answers, predicted_answers)
rogue_1_score = rogue_1_metric(expected_answers, predicted_answers)
rogue_l_score = rogue_l_metric(expected_answers, predicted_answers)
# factscore_score = factscore_metric(expected_answers, predicted_answers)
# summac_zs_score = summac_zs_metric(expected_answers, predicted_answers)
# summac_conv_score = summac_conv_metric(expected_answers, predicted_answers)

# cos_score, bleu_score, rogue_1_score, rogue_l_score, factscore_score, summac_zs_score, summac_conv_score
cos_score, bleu_score, rogue_1_score, rogue_l_score

3it [00:00,  6.57it/s]

[2026-02-16 21:41:23.608] ---GENERATE STEP-BACK QUERY---
[2026-02-16 21:41:27.194] ---GENERATE SUBQUERIES---
[2026-02-16 21:41:31.778] ---DETERMINE SPECIALIZED SOURCES---
[2026-02-16 21:41:32.679] ---SELECTED SOURCES: ['vectorstore', 'pubmed']---
[2026-02-16 21:41:32.679] ---ROUTE QUESTION---
[2026-02-16 21:41:32.680] ---GENERATE HYDE DOCUMENTS---
[2026-02-16 21:41:35.901][2026-02-16 21:41:35.902] ---RETRIEVE FROM VECTOR STORE---
 ---RETRIEVE FROM PUBMED---
[2026-02-16 21:41:36.632] pub_med_retriever_node HTTP Error 429: Too Many Requests
[2026-02-16 21:41:37.955] ---GRADE DOCUMENTs---
[2026-02-16 21:41:37.955] ---AFTER EXACT DEDUPLICATION: 11 documents---
[2026-02-16 21:41:37.956] ---BM25 TOP CANDIDATES: 10 documents---
[2026-02-16 21:43:10.432] ---FINAL DOCUMENTS NUMBER: 0---
[2026-02-16 21:43:10.432] ---ASSESS GRADED DOCUMENTS---
[2026-02-16 21:43:10.432] ---DECISION: SOME DOCUMENTS ARE NOT RELEVANT TO QUESTION, INCLUDE WEB SEARCH---
[2026-02-16 21:43:10.433] ---WEB SEARCH---
[2026-

4it [02:22, 53.27s/it]

[2026-02-16 21:43:46.001] ---DECISION: GENERATION ADDRESSES QUESTION---
[2026-02-16 21:43:46.117] ---GENERATE STEP-BACK QUERY---
[2026-02-16 21:43:47.374] ---GENERATE SUBQUERIES---
[2026-02-16 21:43:49.668] ---DETERMINE SPECIALIZED SOURCES---
[2026-02-16 21:44:00.666] ---SELECTED SOURCES: ['vectorstore', 'pubmed']---
[2026-02-16 21:44:00.667] ---ROUTE QUESTION---
[2026-02-16 21:44:00.668] ---GENERATE HYDE DOCUMENTS---
[2026-02-16 21:44:05.648][2026-02-16 21:44:05.649] ---RETRIEVE FROM VECTOR STORE---
 ---RETRIEVE FROM PUBMED---
[2026-02-16 21:44:06.391] pub_med_retriever_node HTTP Error 429: Too Many Requests
[2026-02-16 21:44:34.213] ---GRADE DOCUMENTs---
[2026-02-16 21:44:34.213] ---AFTER EXACT DEDUPLICATION: 11 documents---
[2026-02-16 21:44:34.216] ---BM25 TOP CANDIDATES: 10 documents---
[2026-02-16 21:44:39.052] ---FINAL DOCUMENTS NUMBER: 0---
[2026-02-16 21:44:39.053] ---ASSESS GRADED DOCUMENTS---
[2026-02-16 21:44:39.053] ---DECISION: SOME DOCUMENTS ARE NOT RELEVANT TO QUESTION,

5it [03:54, 66.50s/it]

...

[2026-02-16 22:12:46.913] ---GRADE GENERATION---
[2026-02-16 22:12:47.013] ---GENERATE STEP-BACK QUERY---
[2026-02-16 22:12:49.411] ---GENERATE SUBQUERIES---
[2026-02-16 22:12:52.349] ---DETERMINE SPECIALIZED SOURCES---
[2026-02-16 22:12:53.311] ---SELECTED SOURCES: ['vectorstore', 'pubmed']---
[2026-02-16 22:12:53.312] ---ROUTE QUESTION---
[2026-02-16 22:12:53.312] ---GENERATE HYDE DOCUMENTS---
[2026-02-16 22:12:57.038][2026-02-16 22:12:57.039] ---RETRIEVE FROM VECTOR STORE---
 ---RETRIEVE FROM PUBMED---
[2026-02-16 22:12:57.672] pub_med_retriever_node HTTP Error 429: Too Many Requests
Too Many Requests, waiting for 26214.40 seconds...
Too Many Requests, waiting for 26214.40 seconds...
Too Many Requests, waiting for 26214.40 seconds...
[2026-02-16 22:14:57.041] pub_med_retriever_node timed out
[2026-02-16 22:14:57.043] ---GRADE DOCUMENTs---
[2026-02-16 22:14:57.043] ---AFTER EXACT DEDUPLICATION: 9 documents---
[2026-02-16 22:14:57.044] ---BM25 TOP CANDIDATES: 9 documents---
[2026-02-1

20it [33:54, 101.72s/it]

[2026-02-16 22:15:17.510] ---DECISION: GENERATION ADDRESSES QUESTION---


(0.7656389024167024,
 0.01491632659156627,
 0.2764323707861091,
 0.2131711549468054)

Too Many Requests, waiting for 419430.40 seconds...
Too Many Requests, waiting for 419430.40 seconds...
Too Many Requests, waiting for 419430.40 seconds...
Too Many Requests, waiting for 13421772.80 seconds...
Too Many Requests, waiting for 13421772.80 seconds...
Too Many Requests, waiting for 13421772.80 seconds...
